read data from bronze layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuit"
silver_table = f"{catalog_name}.{silver_schema}.circuit"

In [0]:
circuit_df = spark.read.table(bronze_table)

keep only column require for analytics drop url column

In [0]:
from pyspark.sql import functions as F

selected_circuit_df = circuit_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_date"),
    F.col("source_file")
)

display(selected_circuit_df)

standardise column with snaka case(countryName ==> country_name)

In [0]:
standardise_circuit_df = selected_circuit_df.withColumnsRenamed({
                                    "circuitId":"circuit_id",
                                    "circuitName":"circuit_name",
                                    "lat":"latitude",
                                    "long":"longitude"
                                })

display(standardise_circuit_df)


#### filter out key where circuit_id is null for bussiness key validation

In [0]:
valid_circuit_df = standardise_circuit_df.filter(F.col("circuit_id").isNotNull())


#### Drop Duplicate Record

In [0]:
circuit_distint_df = valid_circuit_df.dropDuplicates(["circuit_id"])

#### transform values of column ciruit_name and locality to title case

In [0]:
circuit_final_df = circuit_distint_df.withColumn("circuit_name",F.initcap(F.col("circuit_name"))) \
                                    .withColumn("locality",F.initcap(F.col("locality")))
                            

#### write data to silver layer

In [0]:
circuit_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))